# MLP en Vision — Clasificacion de Prendas (Fashion-MNIST)

| Campo | Valor |
|:------|:------|
| **Autor** | Borja Mora Mendez |
| **Contacto** | borja.mora.mendez@gmail.com |
| **LinkedIn** | [linkedin.com/in/borjamoramendez](https://linkedin.com/in/borjamoramendez) |
| **Categoria** | Machine Learning > Redes Neuronales > MLP (Vision) |
| **Dataset** | Fashion-MNIST (70.000 imagenes 28x28, Zalando Research) |
| **Ultima actualizacion** | Julio 2026 |

## 1. Contexto de negocio

Un marketplace de moda online necesita clasificar automaticamente las fotos
de productos que los vendedores suben a la plataforma. La categorizacion
manual genera:

- **Errores:** 8% de productos mal etiquetados, provocando devoluciones.
- **Coste:** 2 operarios dedicados full-time a clasificar imagenes.
- **Latencia:** 24-48h desde subida hasta publicacion del producto.

Pregunta analitica: **puede un MLP clasificar imagenes de prendas con
precision suficiente para automatizar el etiquetado, o necesitamos
una arquitectura convolucional (CNN)?**

## 2. Objetivo y justificacion del modelo

Este notebook evalua las **limitaciones del MLP en vision** comparandolo con
un modelo CNN basico. El MLP trata cada pixel como feature independiente,
perdiendo la informacion espacial que las CNNs capturan nativamente.

| Modelo | Ventaja | Limitacion |
|:-------|:--------|:-----------|
| MLP (flatten) | Simple, rapido de entrenar | Pierde estructura espacial |
| CNN (Conv2D) | Captura patrones visuales | Mas parametros, mas lento |
| Random Forest | No requiere GPU | No escala a imagenes grandes |

In [ ]:
# -- Datos --
import numpy as np
import pandas as pd

# -- Visualizacion --
import matplotlib.pyplot as plt

# -- Deep Learning --
import tensorflow as tf
from tensorflow import keras

# -- ML clasico --
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print(f'TensorFlow: {tf.__version__}')
print('Entorno configurado correctamente.')

## 4. Carga y exploracion inicial

### Diccionario de clases

| ID | Clase | Descripcion |
|:---|:------|:------------|
| 0 | T-shirt/top | Camisetas |
| 1 | Trouser | Pantalones |
| 2 | Pullover | Jerseys |
| 3 | Dress | Vestidos |
| 4 | Coat | Abrigos |
| 5 | Sandal | Sandalias |
| 6 | Shirt | Camisas |
| 7 | Sneaker | Zapatillas |
| 8 | Bag | Bolsos |
| 9 | Ankle boot | Botines |

In [ ]:
(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.fashion_mnist.load_data()

CLASES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
          'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f'Train: {X_train_raw.shape} | Test: {X_test_raw.shape}')
print(f'Rango pixeles: [{X_train_raw.min()}, {X_train_raw.max()}]')
print(f'\nDistribucion del target (train):')
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {CLASES[u]:12s}: {c:5d} ({c/len(y_train):.1%})')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
np.random.seed(42)
for ax, idx in zip(axes.flat, np.random.choice(len(X_train_raw), 10, replace=False)):
    ax.imshow(X_train_raw[idx], cmap='gray')
    ax.set_title(CLASES[y_train[idx]], fontsize=10)
    ax.axis('off')
plt.suptitle('Muestra del Dataset Fashion-MNIST', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 5. Preparacion de datos

In [ ]:
X_train_flat = X_train_raw.reshape(-1, 784).astype('float32')
X_test_flat  = X_test_raw.reshape(-1, 784).astype('float32')

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_flat)
X_test_sc  = scaler.transform(X_test_flat)

X_train_cnn = X_train_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test_cnn  = X_test_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0

print(f'MLP input: {X_train_sc.shape}')
print(f'CNN input: {X_train_cnn.shape}')

## 6. Modelo principal: MLP (sklearn)

El MLP trata cada pixel como feature independiente. Con 784 features y
60.000 muestras, evaluamos si las relaciones pixel-pixel son suficientes
sin considerar la estructura espacial de la imagen.

In [ ]:
np.random.seed(42)
idx_train = np.random.choice(len(X_train_sc), 5000, replace=False)
idx_test  = np.random.choice(len(X_test_sc), 1000, replace=False)

mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    solver='adam',
    max_iter=30,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    batch_size=256
)
mlp.fit(X_train_sc[idx_train], y_train[idx_train])
y_pred_mlp = mlp.predict(X_test_sc[idx_test])
acc_mlp = accuracy_score(y_test[idx_test], y_pred_mlp)
print(f'Accuracy MLP (sklearn, 5k muestra): {acc_mlp:.1%}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test[idx_test], y_pred_mlp,
    display_labels=CLASES, cmap='Blues', ax=ax,
    xticks_rotation=45
)
ax.set_title('Matriz de Confusion - MLP (flatten)', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Modelo alternativo: CNN (Keras)

La CNN usa capas Conv2D que explotan la estructura espacial 2D de la imagen:
filtros 3x3 detectan bordes, texturas y patrones locales.

In [ ]:
tf.random.set_seed(42)
cnn = keras.Sequential([
    keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = cnn.fit(X_train_cnn, y_train, epochs=3, batch_size=256,
                  validation_split=0.1, verbose=1)
_, acc_cnn = cnn.evaluate(X_test_cnn, y_test, verbose=0)
print(f'\nAccuracy CNN (3 epochs): {acc_cnn:.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('Loss - CNN', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Val')
axes[1].set_title('Accuracy - CNN', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

## 8. Modelo alternativo: Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(X_train_flat[idx_train], y_train[idx_train])
y_pred_rf = rf.predict(X_test_flat[idx_test])
acc_rf = accuracy_score(y_test[idx_test], y_pred_rf)
print(f'Accuracy Random Forest (5k muestra): {acc_rf:.1%}')

## 9. Comparativa visual

In [ ]:
y_pred_cnn = np.argmax(cnn.predict(X_test_cnn[idx_test], verbose=0), axis=1)
errores_mlp = np.where(y_pred_mlp != y_test[idx_test])[0]

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
np.random.seed(42)
sample_err = np.random.choice(errores_mlp, min(10, len(errores_mlp)), replace=False)
for ax, e in zip(axes.flat, sample_err):
    ax.imshow(X_test_raw[idx_test[e]], cmap='gray')
    real = CLASES[y_test[idx_test[e]]]
    pred_m = CLASES[y_pred_mlp[e]]
    pred_c = CLASES[y_pred_cnn[e]]
    ax.set_title(f'Real: {real}\nMLP: {pred_m}\nCNN: {pred_c}',
                 fontsize=8, color='red' if pred_m != real else 'black')
    ax.axis('off')
plt.suptitle('Errores del MLP (comparados con CNN)', fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
models = ['MLP (flatten)', 'CNN (Conv2D)', 'Random Forest']
accs = [acc_mlp, acc_cnn, acc_rf]
colors = ['steelblue', 'darkorange', 'forestgreen']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(models, accs, color=colors, alpha=0.85)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.1%}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0.5, 1.0)
ax.set_title('Accuracy Comparativa - Vision (Fashion-MNIST)', fontsize=13)
ax.set_ylabel('Accuracy')
plt.tight_layout()
plt.show()

## 10. Tabla comparativa

| Modelo | Accuracy | Estructura espacial | Velocidad |
|:-------|:---------|:--------------------|:----------|
| MLP (flatten) | Ver resultados | No la captura | Rapido |
| CNN (Conv2D) | Ver resultados | Si (filtros 3x3) | Medio |
| Random Forest | Ver resultados | No la captura | Lento con muchas features |

La CNN supera al MLP porque los filtros convolucionales detectan patrones
locales (bordes, texturas) que el MLP no puede aprender al tratar cada pixel
de forma independiente.

## 11. Insights y recomendaciones accionables

### Hallazgo clave
El MLP pierde informacion espacial critica al aplanar la imagen. Las confusiones
tipicas (pullover vs coat, shirt vs t-shirt) involucran prendas que difieren
en detalles de forma que solo los filtros convolucionales detectan.

### Recomendaciones

**1. Usar CNN como modelo de produccion (impacto: alto, esfuerzo: bajo)**
Incluso una CNN basica de 2 capas supera al MLP. Para el marketplace,
cada punto porcentual reduce devoluciones por error de etiquetado.

**2. Transfer learning con MobileNet (impacto: alto, esfuerzo: medio)**
Para imagenes reales de producto (no 28x28 grayscale), usar un modelo
preentrenado como MobileNet con fine-tuning en las categorias del catalogo.

**3. Data augmentation para clases dificiles (impacto: medio, esfuerzo: bajo)**
Las clases shirt/t-shirt/pullover se confunden. Aplicar rotacion, flip y
zoom para enriquecer los ejemplos de entrenamiento de estas categorias.